# Chapter 70: AI-Powered Threat Detection

**Volume 4 — Security Operations**

**The attacker is already inside.** They authenticated with real credentials.
They move with RDP — the same tool your admins use. The firewall sees nothing wrong.
The SIEM rules fire on nothing. But AI sees what rules cannot: **behavioral deviation**.

### What You Will Build — 5 Detection Engines

| Demo | Technique | What It Catches |
|------|-----------|----------------|
| **1. UEBA Baseline + Scorer** | Weighted anomaly scoring | Insider threat / lateral movement |
| **2. Access Graph Analyzer** | Graph edge novelty detection | Lateral movement traversal pattern |
| **3. C2 Beacon Detector** | Periodicity / CV analysis | Command & Control beaconing |
| **4. DGA Domain Classifier** | Entropy + n-gram scoring | Malware C2 domain generation |
| **5. Full SOC Triage Pipeline** | All engines + LLM analyst | Alert enrichment + MITRE ATT&CK |

### The Core Insight
> **Signature detection matches known bad patterns — and misses everything new.
> Behavioral analytics asks: "Is this entity acting like itself today?"
> The answer to that question catches zero-days, insider threats, and living-off-the-land attacks
> that no signature will ever find.**

In [ ]:
# pip install anthropic
# AI-Powered Threat Detection — pure Python, no ML libraries required!

import os, json, math, time, random, hashlib, re
from collections import defaultdict, deque
from dataclasses import dataclass, field
from typing import List, Dict, Optional

# ── Anthropic client ───────────────────────────────────────────────────────────
api_key = os.environ.get("ANTHROPIC_API_KEY", "")
USE_API = bool(api_key)

if USE_API:
    from anthropic import Anthropic
    client = Anthropic()
    print("LLM analyst: ACTIVE (Anthropic API)")
else:
    print("LLM analyst: SIMULATION MODE (set ANTHROPIC_API_KEY for real analysis)")

def llm_analyze(prompt: str, max_tokens: int = 200) -> str:
    '''Call Anthropic API or return simulation.'''
    if USE_API:
        resp = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=max_tokens,
            messages=[{"role": "user", "content": prompt}]
        )
        return resp.content[0].text.strip()
    # Simulation: extract key info from prompt and return plausible analysis
    if "lateral movement" in prompt.lower() or "new host" in prompt.lower():
        return ("MITRE T1021 (Remote Services) - High confidence lateral movement. "
                "Recommend: isolate account, escalate to IR team, preserve auth logs.")
    if "beacon" in prompt.lower() or "periodic" in prompt.lower():
        return ("MITRE T1071 (Application Layer Protocol) - C2 beaconing detected. "
                "Recommend: block destination IP, memory forensics on source host.")
    if "dga" in prompt.lower() or "entropy" in prompt.lower():
        return ("MITRE T1568.002 (DGA) - Domain Generation Algorithm traffic. "
                "Recommend: DNS sinkhole, full malware scan on querying host.")
    return "Anomalous behavior detected. Recommend analyst review within 30 minutes."

print("Setup complete. All 5 detection engines ready.")


## Demo 1: UEBA Behavioral Baseline + Anomaly Scorer

**UEBA** (User and Entity Behavior Analytics) builds a profile of what "normal" looks like
for each user — which hosts they access, which hours they work, how many systems per session.

When an authentication event deviates from the baseline, it scores high.
Three independent factors combine into one weighted anomaly score:

| Factor | Weight | What It Measures |
|--------|--------|-----------------|
| **New host** | 40% | Has this user ever accessed this host before? |
| **Odd hour** | 30% | Is this outside their normal working hours? |
| **High velocity** | 30% | Too many hosts in one session vs their average? |

A score above **0.6** triggers LLM triage. Below 0.3 = low priority, auto-cleared.

*Network analogy: UEBA is like BGP path history. You know AS65001 only ever announces
10.0.0.0/8. If it suddenly announces 172.16.0.0/12 — valid BGP, no rule broken —
your historical baseline flags it. Same logic: not wrong by the rules, wrong for this entity.*

In [ ]:
# ── Demo 1: UEBA Behavioral Baseline + Anomaly Scorer ─────────────────────────

# Simulated baseline — in production: built from 30-90 days of real auth logs
BASELINES = {
    "alice": {
        "normal_hosts": {"web-01", "repo-01", "dev-db-01"},
        "normal_hours": set(range(8, 19)),      # 08:00 - 18:00
        "avg_daily_hosts": 3,
        "department": "Engineering",
    },
    "bob": {
        "normal_hosts": {"finance-db-01", "payroll-01", "erp-01"},
        "normal_hours": set(range(9, 18)),
        "avg_daily_hosts": 2,
        "department": "Finance",
    },
    "carol": {
        "normal_hosts": {"dc-01", "dc-02", "mgmt-01", "vpn-gw"},
        "normal_hours": set(range(7, 20)),
        "avg_daily_hosts": 5,
        "department": "IT Ops",
    },
}

@dataclass
class AuthEvent:
    '''A single authentication event from Windows Event Log / RADIUS / SSH.'''
    user: str
    host: str
    hour: int           # 0-23
    session_hosts: list # all hosts accessed in this session so far

def ueba_score(event: AuthEvent) -> dict:
    '''Score an auth event against the user behavioral baseline.
    Returns combined score (0.0 = normal, 1.0 = maximum anomaly) + factor breakdown.
    '''
    profile = BASELINES.get(event.user, {})
    factors = {}

    # Factor 1: New host (never seen in baseline) - weight 40%
    known_hosts = profile.get("normal_hosts", set())
    factors["new_host"] = 0.0 if event.host in known_hosts else 0.8

    # Factor 2: Outside normal working hours - weight 30%
    normal_hours = profile.get("normal_hours", set(range(0, 24)))
    factors["odd_hour"] = 0.0 if event.hour in normal_hours else 0.7

    # Factor 3: High velocity (too many hosts vs daily average) - weight 30%
    avg = profile.get("avg_daily_hosts", 3)
    velocity_ratio = len(event.session_hosts) / max(avg, 1)
    factors["high_velocity"] = min(velocity_ratio / 3.0, 1.0)  # cap at 1.0

    # Weighted combination
    weights = {"new_host": 0.40, "odd_hour": 0.30, "high_velocity": 0.30}
    combined = sum(factors[k] * weights[k] for k in factors)

    return {"score": round(combined, 3), "factors": factors,
            "department": profile.get("department", "Unknown")}

def triage_with_llm(event: AuthEvent, scoring: dict) -> str:
    '''For high-score alerts: send to LLM analyst for MITRE ATT&CK mapping.'''
    if scoring["score"] < 0.30:
        return f"  -> LOW ({scoring['score']:.2f}) — auto-cleared, likely normal activity"

    prompt = (
        f"Security alert — behavioral anomaly:\n"
        f"User: {event.user} ({scoring['department']})\n"
        f"Accessed host: {event.host} at {event.hour:02d}:00\n"
        f"Session hosts so far: {event.session_hosts}\n"
        f"Anomaly score: {scoring['score']} | Factors: {scoring['factors']}\n"
        f"User's normal hosts: {list(BASELINES.get(event.user, {}).get('normal_hosts', []))}\n\n"
        f"Is this consistent with lateral movement? Which MITRE ATT&CK technique? "
        f"What immediate action? Keep under 80 words."
    )
    analysis = llm_analyze(prompt, max_tokens=120)
    level = "CRITICAL" if scoring["score"] >= 0.7 else "HIGH"
    return f"  -> {level} ({scoring['score']:.2f}) | {analysis}"

# ── Test events ────────────────────────────────────────────────────────────────
print("=" * 60)
print("UEBA ANOMALY SCORER — AUTH EVENT ANALYSIS")
print("=" * 60)

test_events = [
    AuthEvent("alice", "dev-db-01",      9,  ["dev-db-01"]),               # normal
    AuthEvent("alice", "finance-db-01",  3,  ["finance-db-01", "payroll-01", "backup-01"]),  # lateral movement
    AuthEvent("bob",   "payroll-01",     10, ["payroll-01"]),               # normal
    AuthEvent("carol", "finance-db-01",  2,  ["finance-db-01", "hr-db-01", "ceo-laptop"]),   # suspicious admin
]

for ev in test_events:
    scoring = ueba_score(ev)
    result  = triage_with_llm(ev, scoring)
    print(f"\nEvent: {ev.user} -> {ev.host} @ {ev.hour:02d}:00")
    print(f"  Factors: new_host={scoring['factors']['new_host']:.1f}  "
          f"odd_hour={scoring['factors']['odd_hour']:.1f}  "
          f"velocity={scoring['factors']['high_velocity']:.2f}")
    print(result)

print("\n[UEBA] High scores automatically escalate to SOC queue.")


## Demo 2: Access Graph Lateral Movement Detector

Lateral movement creates **new edges in the access graph** — authentications between
nodes that have never been connected in the historical baseline.

The access graph stores every (user, host) pair seen in the past 90 days.
When a new pair appears, it generates an edge novelty event.
Sequential velocity (new hosts per time window) amplifies the score.

```
Historical graph edges (known normal):
  alice -> web-01, repo-01, dev-db-01
  bob   -> finance-db-01, payroll-01

New edges detected tonight:
  alice -> finance-db-01  (NEW EDGE — never seen)
  alice -> payroll-01     (NEW EDGE — never seen)
  alice -> backup-01      (NEW EDGE — never seen)
  3 new edges in 18 minutes = lateral movement pattern
```

The score formula:
- Base score per new edge: **0.5**
- Velocity multiplier: `new_edges_in_window / 3` (capped at 2.0x)
- Time-of-day bonus: +0.2 if outside business hours
- Final score capped at 1.0

In [ ]:
# ── Demo 2: Access Graph Lateral Movement Detector ────────────────────────────

class AccessGraph:
    '''Tracks historical (user, host) access pairs to detect new lateral movement edges.'''

    def __init__(self):
        # Historical baseline edges: set of (user, host) tuples seen in past 90 days
        self.known_edges: set = set()
        # Recent window: list of (timestamp, user, host) for velocity calculation
        self.recent_events: deque = deque(maxlen=500)
        # Detection log
        self.alerts: list = []

    def load_baseline(self, history: list):
        '''Load 90-day historical access pairs into the known edge set.'''
        for user, host in history:
            self.known_edges.add((user, host))
        print(f"[AccessGraph] Baseline loaded: {len(self.known_edges)} known edges")

    def observe(self, user: str, host: str, timestamp: float, hour: int) -> Optional[dict]:
        '''
        Observe a new auth event. Returns alert dict if new edge detected, else None.
        timestamp: Unix timestamp for velocity window (use time.time() in production)
        hour: 0-23 for time-of-day scoring
        '''
        self.recent_events.append((timestamp, user, host))
        edge = (user, host)

        if edge in self.known_edges:
            return None  # Known edge, no alert

        # New edge detected!
        self.known_edges.add(edge)  # add so we don't re-alert on same pair

        # Count new edges from this user in the last 30 minutes
        window_start = timestamp - 1800  # 30-minute window
        new_in_window = sum(
            1 for ts, u, h in self.recent_events
            if u == user and ts >= window_start and (u, h) not in set()
        )
        # Simpler: count recent events from this user (approximation)
        recent_user = sum(1 for ts, u, h in self.recent_events
                          if u == user and ts >= window_start)

        # Score: base + velocity multiplier + off-hours bonus
        base_score = 0.50
        velocity_mult = min(recent_user / 3.0, 2.0)
        off_hours_bonus = 0.20 if hour not in range(8, 19) else 0.0
        score = min(base_score * velocity_mult + off_hours_bonus, 1.0)

        alert = {
            "type": "LATERAL_MOVEMENT",
            "user": user,
            "new_host": host,
            "hour": hour,
            "recent_events_30min": recent_user,
            "score": round(score, 3),
        }
        self.alerts.append(alert)
        return alert

# ── Build the graph from 90-day baseline ──────────────────────────────────────
graph = AccessGraph()

historical_access = [
    ("alice", "web-01"), ("alice", "repo-01"), ("alice", "dev-db-01"),
    ("bob",   "finance-db-01"), ("bob", "payroll-01"), ("bob", "erp-01"),
    ("carol", "dc-01"), ("carol", "dc-02"), ("carol", "mgmt-01"), ("carol", "vpn-gw"),
    ("dave",  "helpdesk-01"), ("dave", "ticket-db"),
]
graph.load_baseline(historical_access)

# ── Simulate tonight's auth events ────────────────────────────────────────────
print("\n" + "=" * 60)
print("ACCESS GRAPH — TONIGHT'S AUTH EVENTS (03:00 - 03:20)")
print("=" * 60)

base_time = time.time()
tonight_events = [
    # alice normal traffic during the day - no alerts expected
    (base_time - 50000, "alice", "web-01",       9),   # normal
    (base_time - 49000, "alice", "dev-db-01",    10),  # normal
    # 03:00 AM - alice appears on finance systems
    (base_time - 60,    "alice", "finance-db-01", 3),  # NEW EDGE
    (base_time - 30,    "alice", "payroll-01",    3),  # NEW EDGE
    (base_time - 10,    "alice", "backup-server", 3),  # NEW EDGE
    # bob normal daytime
    (base_time - 45000, "bob",   "payroll-01",   10),  # normal
]

for ts, user, host, hour in tonight_events:
    alert = graph.observe(user, host, ts, hour)
    if alert:
        print(f"\n[ALERT] New edge: {user} -> {host} @ {hour:02d}:00")
        print(f"  Score: {alert['score']:.3f}  "
              f"Recent events (30min): {alert['recent_events_30min']}")
        # LLM triage for high scores
        if alert["score"] >= 0.5:
            analysis = llm_analyze(
                f"Lateral movement alert: user {user} accessed new host {host} "
                f"at {hour:02d}:00. Score {alert['score']}. "
                f"This is new edge {alert['recent_events_30min']} in 30 minutes. "
                f"MITRE technique? Immediate action? Under 60 words.",
                max_tokens=80
            )
            print(f"  LLM: {analysis}")
    else:
        print(f"  [OK] {user} -> {host} @ {hour:02d}:00 (known edge, no alert)")

print(f"\n[AccessGraph] Total alerts generated: {len(graph.alerts)}")


## Demo 3: C2 Beacon Detector

**Command and Control (C2) malware** must periodically "phone home" to the attacker.
This creates **beaconing** — traffic that arrives at eerily regular intervals.

Humans generate irregular traffic. Malware generates machine-precise timing.

**Detection metric: Coefficient of Variation (CV)**
```
CV = standard_deviation / mean_inter_arrival_time

Human traffic:    CV ≈ 1.5 - 3.0  (very irregular)
C2 beacon:        CV < 0.15        (suspiciously regular)
Jittered beacon:  CV ≈ 0.10 - 0.25 (malware adds random jitter to evade this)
```

We also simulate **JA3 fingerprint matching** — known malware TLS fingerprints
that match against a threat intelligence feed:

| JA3 Hash | Tool | Threat Level |
|----------|------|-------------|
| `769,47-53...` | Cobalt Strike | CRITICAL |
| `771,4865...`  | Metasploit    | CRITICAL |
| `769,0-35...`  | Sliver C2     | HIGH |

*Network analogy: C2 beacon detection is exactly like detecting a misbehaving
OSPF Hello. Packets arriving at exactly 10-second intervals from an external IP
to an internal host — that machine-like precision is the fingerprint of malware.*

In [ ]:
# ── Demo 3: C2 Beacon Detector ────────────────────────────────────────────────
import statistics

# Simulated JA3 threat intelligence feed (in production: pull from threat intel API)
KNOWN_C2_JA3 = {
    "769,47-53-5-10-49161-49162-49171-49172-50-56-19-4,0-10-11,23-24-25,0":
        {"tool": "Cobalt Strike", "severity": "CRITICAL"},
    "771,4865-4866-4867-49195-49199-52393-52392-49196-49200-49171-49172-156-157-47-53,0-23-65281-10-11-35-16-5-13-18-51-45-43-27-21,29-23-24,0":
        {"tool": "Metasploit Meterpreter", "severity": "CRITICAL"},
    "769,0-35-23-65281,0-11-10,23,0":
        {"tool": "Sliver C2 Framework", "severity": "HIGH"},
}

@dataclass
class NetFlow:
    '''A connection record from NetFlow/IPFIX telemetry.'''
    src_ip: str
    dst_ip: str
    dst_port: int
    timestamps: List[float]   # list of connection timestamps
    ja3_hash: str = ""        # TLS ClientHello fingerprint (empty if not TLS)
    bytes_sent: int = 0

def compute_cv(timestamps: list) -> float:
    '''
    Compute coefficient of variation of inter-arrival times.
    Low CV (<0.15) = machine-like regularity = potential beacon.
    '''
    if len(timestamps) < 4:
        return 999.0  # not enough data

    intervals = [timestamps[i+1] - timestamps[i]
                 for i in range(len(timestamps) - 1)]
    mean = statistics.mean(intervals)
    if mean == 0:
        return 0.0
    std  = statistics.stdev(intervals)
    return std / mean

def detect_beacon(flow: NetFlow) -> Optional[dict]:
    '''Analyze a NetFlow record for C2 beacon behavior.'''
    cv = compute_cv(flow.timestamps)
    mean_interval = 0.0
    if len(flow.timestamps) >= 2:
        intervals = [flow.timestamps[i+1] - flow.timestamps[i]
                     for i in range(len(flow.timestamps) - 1)]
        mean_interval = statistics.mean(intervals)

    alerts = []

    # Check 1: Periodicity (low CV = suspicious)
    if cv < 0.15 and len(flow.timestamps) >= 4:
        alerts.append({
            "type": "PERIODIC_BEACON",
            "cv": round(cv, 4),
            "mean_interval_sec": round(mean_interval, 1),
            "connections": len(flow.timestamps),
        })

    # Check 2: JA3 fingerprint match against known C2 tools
    if flow.ja3_hash and flow.ja3_hash in KNOWN_C2_JA3:
        threat = KNOWN_C2_JA3[flow.ja3_hash]
        alerts.append({
            "type": "KNOWN_C2_JA3",
            "tool": threat["tool"],
            "severity": threat["severity"],
            "ja3": flow.ja3_hash[:40] + "...",
        })

    if not alerts:
        return None

    return {
        "src": flow.src_ip,
        "dst": f"{flow.dst_ip}:{flow.dst_port}",
        "ja3_hash": flow.ja3_hash[:20] + "..." if flow.ja3_hash else "none",
        "cv": round(cv, 4),
        "alerts": alerts,
    }

# ── Simulate NetFlow records ───────────────────────────────────────────────────
print("=" * 60)
print("C2 BEACON DETECTOR — NETFLOW ANALYSIS")
print("=" * 60)

now = time.time()

flows = [
    # Normal web browsing — irregular timing
    NetFlow("10.0.1.50", "8.8.8.8", 443,
            timestamps=[now, now+45, now+312, now+28, now+891, now+1100],
            ja3_hash="", bytes_sent=15000),

    # C2 beacon — 300-second interval, very low CV
    NetFlow("10.0.1.77", "185.220.101.45", 443,
            timestamps=[now, now+300, now+600, now+900, now+1200, now+1500, now+1800],
            ja3_hash="", bytes_sent=1200),

    # Cobalt Strike by JA3 fingerprint
    NetFlow("10.0.2.33", "91.108.4.12", 443,
            timestamps=[now, now+600, now+1210, now+1820],
            ja3_hash="769,47-53-5-10-49161-49162-49171-49172-50-56-19-4,0-10-11,23-24-25,0",
            bytes_sent=4800),

    # Jittered beacon (malware adds +/-15s jitter) — slightly higher CV but still low
    NetFlow("10.0.3.12", "203.0.113.99", 80,
            timestamps=[now, now+302, now+598, now+914, now+1195, now+1512],
            ja3_hash="", bytes_sent=800),
]

for flow in flows:
    result = detect_beacon(flow)
    cv = compute_cv(flow.timestamps)

    if result:
        print(f"\n[BEACON ALERT] {result['src']} -> {result['dst']}")
        print(f"  CV={result['cv']:.4f} | Connections={len(flow.timestamps)}")
        for a in result["alerts"]:
            if a["type"] == "PERIODIC_BEACON":
                print(f"  [!] PERIODIC: interval={a['mean_interval_sec']:.0f}s  CV={a['cv']:.4f}")
            else:
                print(f"  [!] JA3 MATCH: {a['tool']} ({a['severity']})")
        analysis = llm_analyze(
            f"C2 beacon detected from {result['src']} to {result['dst']}. "
            f"CV={result['cv']:.4f} (low = machine-like). Alerts: {result['alerts']}. "
            f"MITRE technique? Recommended action? Under 60 words.",
            max_tokens=80
        )
        print(f"  LLM: {analysis}")
    else:
        print(f"  [OK] {flow.src_ip} -> {flow.dst_ip}:{flow.dst_port} "
              f"CV={cv:.3f} — normal traffic pattern")

print("\n[BeaconDetector] Analysis complete. Alerts routed to SOC queue.")


## Demo 4: DGA Domain Classifier

**Domain Generation Algorithms (DGAs)** are how malware avoids static blocklists.
Instead of hardcoding `evil.com`, the malware generates hundreds of random domains
and connects to whichever one the attacker has registered.

DGA domains are distinguishable from legitimate domains by **three statistical properties**:

**1. Shannon Entropy** — legitimate domains use human-readable words (low entropy).
DGA domains use algorithmically random characters (high entropy).
```
google.com   entropy=3.12   LOW  (recognizable word)
xkqpzbmvow.com entropy=3.45  HIGH (random-looking)
```

**2. N-gram frequency** — legitimate domains contain common English letter pairs
(bi-grams like "er", "in", "th"). DGA domains have rare, random bi-gram sequences.

**3. Domain length** — DGA domains cluster at specific length ranges
depending on the algorithm; outliers (>20 chars) are suspicious.

All three signals combine into a DGA probability score.
The LLM adds context: why is this host querying `.ru` TLDs at 3am?

In [ ]:
# ── Demo 4: DGA Domain Classifier ────────────────────────────────────────────

# Common English bigrams (letter pairs) — high frequency in real words
COMMON_BIGRAMS = {
    "th","he","in","er","an","re","on","at","en","nd","ti","es","or","te",
    "of","ed","is","it","al","ar","st","to","nt","ng","se","ha","as","ou",
    "io","le","ve","co","me","de","hi","ri","ro","ic","ne","ea","ra","ce",
    "li","ch","ll","be","ma","si","om","ur","ca","el","ta","la","ns","di",
    "fo","ho","pe","ec","pr","no","ct","us","tr","ss","ow","rs","un","ly",
}

def shannon_entropy(s: str) -> float:
    '''Calculate Shannon entropy of a string. Higher = more random.'''
    if not s:
        return 0.0
    freq = defaultdict(int)
    for c in s:
        freq[c] += 1
    length = len(s)
    entropy = 0.0
    for count in freq.values():
        p = count / length
        entropy -= p * math.log2(p)
    return entropy

def bigram_score(domain_label: str) -> float:
    '''
    Fraction of bigrams in the domain label that are common English pairs.
    High score (>0.4) = legitimate. Low score (<0.2) = DGA-like.
    '''
    if len(domain_label) < 2:
        return 1.0
    bigrams = [domain_label[i:i+2] for i in range(len(domain_label) - 1)]
    if not bigrams:
        return 0.0
    common_count = sum(1 for bg in bigrams if bg in COMMON_BIGRAMS)
    return common_count / len(bigrams)

def dga_score(fqdn: str) -> dict:
    '''
    Score a domain for DGA likelihood.
    Returns: probability (0.0=legit, 1.0=DGA), and factor breakdown.
    '''
    # Extract the registered domain label (e.g., "google" from "google.com")
    parts = fqdn.lower().rstrip(".").split(".")
    if len(parts) < 2:
        label = fqdn
    else:
        label = parts[-2]  # second-level domain

    entropy    = shannon_entropy(label)
    bigram_frac = bigram_score(label)
    label_len  = len(label)

    # Factor scores (0.0 = normal, 1.0 = maximum suspicion)
    # Entropy: >3.5 bits/char for short labels is suspicious
    entropy_score = min(max((entropy - 2.8) / 1.5, 0.0), 1.0)

    # Bigram: very few common pairs
    bigram_suspicion = 1.0 - bigram_frac

    # Length: 15-30 chars is common DGA range; >25 very suspicious
    if label_len > 25:
        length_score = 1.0
    elif label_len > 15:
        length_score = (label_len - 15) / 10.0
    else:
        length_score = 0.0

    # Weighted combination
    dga_probability = (
        entropy_score    * 0.45 +
        bigram_suspicion * 0.40 +
        length_score     * 0.15
    )

    return {
        "domain": fqdn,
        "label": label,
        "dga_prob": round(min(dga_probability, 1.0), 3),
        "entropy": round(entropy, 3),
        "bigram_score": round(bigram_frac, 3),
        "label_len": label_len,
    }

# ── Test domains ───────────────────────────────────────────────────────────────
print("=" * 60)
print("DGA DOMAIN CLASSIFIER — DNS LOG ANALYSIS")
print("=" * 60)

test_domains = [
    # Legitimate domains
    ("microsoft.com",        "legit SaaS"),
    ("googleusercontent.com","legit CDN"),
    ("stackoverflow.com",    "legit dev resource"),
    # DGA-generated domains (algorithmically random)
    ("xkqpzbmvowf.ru",       "DGA - high entropy random"),
    ("q7h2mxvzlpntk.net",    "DGA - numeric+random"),
    ("aabbccddeeffgghhii.biz","DGA - long repetitive"),
    ("wnsxhqmtpjzr.com",     "DGA - consonant-heavy"),
    # Borderline
    ("cloudflare.com",       "legit - low entropy"),
    ("akamaitechnologies.com","legit - long but pronounceable"),
]

print(f"{'Domain':<35} {'Prob':>6} {'Entropy':>8} {'Bigram':>8} {'Verdict'}")
print("-" * 75)

flagged = []
for domain, label in test_domains:
    result = dga_score(domain)
    verdict = "DGA" if result["dga_prob"] >= 0.55 else ("BORDERLINE" if result["dga_prob"] >= 0.35 else "LEGIT")
    flag = " <-- ALERT" if verdict == "DGA" else ""
    print(f"{domain:<35} {result['dga_prob']:>6.3f} {result['entropy']:>8.3f} "
          f"{result['bigram_score']:>8.3f} {verdict}{flag}")
    if verdict == "DGA":
        flagged.append(result)

# LLM enrichment for flagged domains
if flagged:
    print(f"\n[DGA] {len(flagged)} domains flagged. Requesting LLM enrichment...")
    for r in flagged[:2]:  # enrich top 2
        analysis = llm_analyze(
            f"DGA detection: domain '{r['domain']}' scored {r['dga_prob']:.2f} DGA probability. "
            f"Entropy={r['entropy']:.2f}, bigram={r['bigram_score']:.2f}. "
            f"Host made 247 queries to this pattern in 2 hours at 3am. "
            f"MITRE T1568.002? Recommended response? Under 60 words.",
            max_tokens=80
        )
        print(f"  [!] {r['domain']}: {analysis}")


## Demo 5: Full SOC Triage Pipeline

**All four detectors wired together** into a complete Security Operations Center
triage pipeline — the way a real AI-augmented SOC works.

```
Auth logs    ->  UEBA Scorer     -+
NetFlow      ->  Beacon Detector  |-> Alert Queue -> LLM Triage -> Priority Report
DNS logs     ->  DGA Classifier  -+
```

**The alert fatigue problem**: a large enterprise SIEM generates 10,000-100,000 alerts/day.
A SOC team can investigate 50-200 per shift. That's a 98-99% uninvestigated gap.

AI-assisted triage solves this in two ways:
1. **Automated enrichment**: context pre-populated before analyst sees the alert
2. **Priority scoring**: high-fidelity detections rise to top; low-confidence deprioritized

The analyst's job changes: not "gather context" but "make the decision" with context ready.

> **Non-negotiable guardrail: All recommended actions require human approval.
> The AI recommends; the analyst decides. No automated blocking without confirmation.**

In [ ]:
# ── Demo 5: Full SOC Triage Pipeline ─────────────────────────────────────────

@dataclass
class ThreatAlert:
    '''A normalized alert from any detection engine.'''
    alert_id: str
    source: str           # UEBA / BEACON / DGA
    severity: str         # CRITICAL / HIGH / MEDIUM / LOW
    score: float          # 0.0 - 1.0
    entity: str           # affected user or host
    description: str
    raw_data: dict
    mitre_technique: str = ""
    analyst_action: str = ""

class SOCTriage:
    '''
    Full SOC triage pipeline.
    Ingests auth events, NetFlow, and DNS logs.
    Runs all detection engines and produces prioritized alert report.
    '''

    def __init__(self):
        self.access_graph = AccessGraph()
        self.alert_queue: List[ThreatAlert] = []
        self._alert_counter = 0

    def _new_id(self) -> str:
        self._alert_counter += 1
        return f"SOC-{self._alert_counter:04d}"

    def ingest_auth(self, events: list):
        '''Process authentication events through UEBA + access graph.'''
        for user, host, hour, session_hosts in events:
            # UEBA score
            ev = AuthEvent(user, host, hour, session_hosts)
            scoring = ueba_score(ev)

            if scoring["score"] >= 0.30:
                sev = "CRITICAL" if scoring["score"] >= 0.7 else "HIGH"
                self.alert_queue.append(ThreatAlert(
                    alert_id=self._new_id(),
                    source="UEBA",
                    severity=sev,
                    score=scoring["score"],
                    entity=user,
                    description=f"{user} accessed {host} at {hour:02d}:00 — anomaly score {scoring['score']:.2f}",
                    raw_data={"event": ev.__dict__, "scoring": scoring},
                ))

    def ingest_netflow(self, flows: list):
        '''Process NetFlow records through beacon detector.'''
        for flow_data in flows:
            flow = NetFlow(**flow_data)
            result = detect_beacon(flow)
            if result:
                highest = max(result["alerts"], key=lambda a: 1 if a.get("severity") == "CRITICAL" else 0)
                sev = highest.get("severity", "HIGH")
                self.alert_queue.append(ThreatAlert(
                    alert_id=self._new_id(),
                    source="BEACON",
                    severity=sev,
                    score=min(1.0 - result["cv"] + 0.5, 1.0) if result["cv"] < 0.5 else 0.6,
                    entity=result["src"],
                    description=f"Beacon from {result['src']} to {result['dst']} | CV={result['cv']:.4f}",
                    raw_data=result,
                ))

    def ingest_dns(self, dns_queries: list):
        '''Process DNS queries through DGA classifier.'''
        for src_ip, domain, count in dns_queries:
            result = dga_score(domain)
            if result["dga_prob"] >= 0.55:
                self.alert_queue.append(ThreatAlert(
                    alert_id=self._new_id(),
                    source="DGA",
                    severity="HIGH",
                    score=result["dga_prob"],
                    entity=src_ip,
                    description=f"{src_ip} queried DGA domain {domain} x{count} times | prob={result['dga_prob']:.2f}",
                    raw_data={"query": result, "count": count},
                ))

    def triage_all(self) -> List[ThreatAlert]:
        '''Enrich all alerts with LLM analysis. Returns sorted priority list.'''
        print(f"\n[SOC] Triaging {len(self.alert_queue)} alerts...")

        for alert in self.alert_queue:
            if alert.severity in ("CRITICAL", "HIGH"):
                prompt = (
                    f"SOC alert [{alert.source}] severity={alert.severity} score={alert.score:.2f}\n"
                    f"Entity: {alert.entity}\n"
                    f"Description: {alert.description}\n"
                    f"Provide: 1) MITRE ATT&CK technique 2) Recommended analyst action. "
                    f"Under 60 words."
                )
                response = llm_analyze(prompt, max_tokens=90)
                # Parse MITRE from response (heuristic)
                mitre = "T1078" if "T10" in response or "T107" in response else "See analysis"
                alert.mitre_technique = response.split(".")[0] if response else "Unknown"
                alert.analyst_action = response

        # Sort by score descending (highest priority first)
        return sorted(self.alert_queue, key=lambda a: a.score, reverse=True)

    def print_report(self, alerts: List[ThreatAlert]):
        '''Print prioritized SOC alert report.'''
        print("\n" + "=" * 65)
        print("SOC TRIAGE REPORT — PRIORITY QUEUE")
        print("=" * 65)
        print(f"{'#':<4} {'ID':<10} {'SRC':<8} {'SEV':<10} {'SCORE':<7} {'ENTITY':<20}")
        print("-" * 65)
        for i, a in enumerate(alerts[:8], 1):
            print(f"{i:<4} {a.alert_id:<10} {a.source:<8} {a.severity:<10} {a.score:<7.3f} {a.entity:<20}")
        print("-" * 65)
        print("\n[TOP PRIORITY DETAILS]")
        for a in alerts[:3]:
            print(f"\n{a.alert_id} [{a.source}] {a.severity} — {a.entity}")
            print(f"  {a.description}")
            if a.analyst_action:
                print(f"  LLM: {a.analyst_action}")
            print(f"  -> ACTION REQUIRED: Analyst must approve before any blocking")

# ── Run the full pipeline ──────────────────────────────────────────────────────
print("=" * 65)
print("FULL SOC TRIAGE PIPELINE — RUNNING ALL DETECTION ENGINES")
print("=" * 65)

soc = SOCTriage()

# Load access graph baseline
soc.access_graph.load_baseline([
    ("alice","web-01"),("alice","repo-01"),("alice","dev-db-01"),
    ("bob","finance-db-01"),("bob","payroll-01"),
])

# Ingest auth events
soc.ingest_auth([
    ("alice", "dev-db-01",      9,  ["dev-db-01"]),                              # normal - no alert
    ("alice", "finance-db-01",  3,  ["finance-db-01","payroll-01","backup-01"]), # lateral movement
    ("bob",   "domain-ctrl-01", 3,  ["domain-ctrl-01","all-users-share"]),       # suspicious admin access
])

# Ingest NetFlow
now = time.time()
soc.ingest_netflow([
    {"src_ip":"10.0.1.77","dst_ip":"185.220.101.45","dst_port":443,
     "timestamps":[now,now+300,now+600,now+900,now+1200,now+1500],
     "ja3_hash":"","bytes_sent":1200},
    {"src_ip":"10.0.2.33","dst_ip":"91.108.4.12","dst_port":443,
     "timestamps":[now,now+600,now+1210,now+1820],
     "ja3_hash":"769,47-53-5-10-49161-49162-49171-49172-50-56-19-4,0-10-11,23-24-25,0",
     "bytes_sent":4800},
])

# Ingest DNS
soc.ingest_dns([
    ("10.0.1.77", "xkqpzbmvowf.ru",    847),
    ("10.0.2.33", "q7h2mxvzlpntk.net", 312),
    ("10.0.0.1",  "microsoft.com",      5),    # legit - no alert
])

# Triage and report
prioritized = soc.triage_all()
soc.print_report(prioritized)

print(f"\n{'='*65}")
print(f"SUMMARY: {len(prioritized)} alerts | "
      f"CRITICAL={sum(1 for a in prioritized if a.severity=='CRITICAL')} | "
      f"HIGH={sum(1 for a in prioritized if a.severity=='HIGH')}")
print("All recommended actions await analyst approval — human-in-the-loop enforced.")


## Summary: What You Built

You now have a working **AI-Powered Threat Detection** system with 5 detection engines:

| Engine | Technique | Key Metric | Threshold |
|--------|-----------|------------|-----------|
| **UEBA Scorer** | Weighted behavioral deviation | Combined score | > 0.60 = alert |
| **Access Graph** | New edge detection | Edge novelty + velocity | Any new edge at odd hours |
| **Beacon Detector** | Periodicity analysis | Coefficient of Variation (CV) | CV < 0.15 = beacon |
| **DGA Classifier** | Entropy + n-gram | DGA probability | > 0.55 = flag |
| **SOC Triage** | LLM enrichment + priority sort | Priority score | Human approval required |

### Why Behavioral Detection > Signatures

- **Signatures**: catch known attacks only. Miss zero-days, living-off-the-land, credential abuse
- **Behavioral**: catches _any_ deviation from normal — doesn't matter if it's a new attack

### The Non-Negotiable Rule

> **Every AI recommendation requires human approval before execution.**
> The AI enriches context and prioritizes. The analyst decides.
> Hallucinated blocklist entries cause outages. Human oversight is not optional.

### Production Upgrade Path

```
Pure Python baseline          ->  30-90 day rolling statistics per user/entity
Simulated LLM triage          ->  Anthropic claude-haiku-4-5-20251001 via API
Hardcoded JA3 lookup          ->  Real-time threat intel feed (VirusTotal, Shodan)
In-memory alert queue         ->  SIEM integration (Splunk, Elastic, QRadar)
Flat file logging             ->  Structured log pipeline + vector search for RAG
```

**Next: Chapter 72 — Security Log Analysis & SIEM Integration** — building the data
pipeline that feeds this detection system with real Cisco ASA, Juniper SRX,
and Windows Event Log telemetry.

In [ ]:
# ── Full Integration: Production-Ready Threat Detection Checklist ─────────────
print("CHAPTER 70: PRODUCTION DEPLOYMENT CHECKLIST")
print("=" * 60)

checklist = [
    # Data ingestion
    ("Data Sources",        "Windows Event Log (4624/4625/4648) -> UEBA"),
    ("Data Sources",        "NetFlow/IPFIX from core routers -> Beacon detector"),
    ("Data Sources",        "DNS query logs (recursive resolver) -> DGA classifier"),
    ("Data Sources",        "RADIUS/Tacacs+ auth logs -> Access graph"),
    # Detection tuning
    ("Detection Tuning",    "UEBA baseline period: 30 days minimum (90 recommended)"),
    ("Detection Tuning",    "Beacon CV threshold: 0.15 (tune per environment)"),
    ("Detection Tuning",    "DGA threshold: 0.55 (lower = more FPs, higher = more FNs)"),
    ("Detection Tuning",    "Access graph: re-baseline after role changes"),
    # LLM integration
    ("LLM Integration",     "Model: claude-haiku-4-5-20251001 (fast, cost-effective)"),
    ("LLM Integration",     "RAG knowledge base: MITRE ATT&CK + internal runbooks"),
    ("LLM Integration",     "Always show reasoning chain — never just 'malicious'"),
    ("LLM Integration",     "Confidence separation: detection% vs attribution%"),
    # Guardrails
    ("Guardrails",          "ALL actions require explicit analyst approval"),
    ("Guardrails",          "Immutable audit log: every AI recommendation + outcome"),
    ("Guardrails",          "Rollback capability: pre-change state snapshot required"),
    ("Guardrails",          "False positive feedback loop: analyst decisions -> retraining"),
]

current_category = ""
for category, item in checklist:
    if category != current_category:
        print(f"\n[{category}]")
        current_category = category
    print(f"  ✓ {item}")

print("\n" + "=" * 60)
print("KEY DETECTION FORMULAS")
print("=" * 60)
print("UEBA score     = 0.40*(new_host) + 0.30*(odd_hour) + 0.30*(velocity)")
print("Beacon CV      = stddev(intervals) / mean(intervals)  ->  < 0.15 = alert")
print("DGA probability= 0.45*(entropy_score) + 0.40*(bigram_suspicion) + 0.15*(length)")
print("Combined score -> alert -> LLM enrichment -> analyst queue -> human decision")
print("\nAll 5 engines run in parallel. Results merge into unified SOC priority queue.")
print("Chapter 70 complete. Build the pipeline. Trust the analyst.")
